# CEGA-Tiny v4: Part B experiments
Cells 0-5 are unchanged from `CEGA_Tiny_v3_Experiments.ipynb`. Cells B0-B5 add: two new streams (SEA, NOAA Weather), two matched-rate baselines (selective backprop, entropy sampling), int8 inference, delayed labels and a solar harvest profile. Cell B1 checks that the extended simulator reproduces v3 exactly. Runtime on a Colab CPU: about 10-15 minutes. At the end a zip `cega_tiny_partB_results.zip` is downloaded.

In [1]:
# ============================== 0. Imports & configuration ==============================
import os, io, sys, json, time, math, zipfile, shutil, warnings, urllib.request
from collections import deque
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg") if "google.colab" not in sys.modules else None
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import f1_score

warnings.filterwarnings("ignore")

SMOKE = os.environ.get("CEGA_SMOKE", "0") == "1"   # quick synthetic self-test only; keep False on Colab
OUT = "cega_tiny_partB_results"
DATA_DIR = "data"
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA_DIR, exist_ok=True)
for sub in ["figures", "tables", "raw"]:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)

CFG = dict(
    seeds=[0, 1, 2, 3, 4],
    window=50,            # samples per stream window
    p_label=0.20,         # fraction of stream samples whose label is revealed (sparse supervision)
    buffer=200,           # FIFO replay buffer capacity (labelled samples)
    min_buffer=40,        # minimum labelled samples before any retraining
    hidden=(64, 32),      # tiny MLP hidden sizes
    pre_epochs=40, pre_lr=0.05,
    adapt_epochs=5,       # fixed epochs used by retraining baselines
    max_epochs=8,         # upper bound for CEGA (early stopping may use fewer)
    lr_adapt=0.02, lr_online=0.01, momentum=0.9, batch=32,
    beta=0.5,             # spare harvested energy per window = beta * E_full / 20
    cap_mult=3.0,         # battery capacity = cap_mult * E_full
    soc0=0.5,             # initial state of charge
    reserve=0.10,         # CEGA never plans below this SoC
    period=20,            # Periodic-Full retraining period (windows)
    lambda0=0.3,          # CEGA energy price
    g_scale=0.10,         # converts normalised energy into accuracy units
    g_prior={1: 0.04, 2: 0.06, 3: 0.08},  # prior expected accuracy gain per adaptation depth
    g_ema=0.3,
    pending_max=10, cooldown=3, recent=3,
    ph_err=dict(delta=0.005, lam=15.0, min_n=30),
    ph_ent=dict(delta=0.01, lam=0.30, min_n=5),
    # ---- CEGA-Tiny v2 (energy-governed gating of on-device online learning) ----
    v2_tau_min=0.50, v2_tau_max=0.95,   # update a labelled sample only if p(true class) < tau; tau grows with SoC
    v2_boost=5,                         # windows of ungated full-depth updates after a detected drift
    v2_burst_epochs=1,                  # replay epochs over the buffer when drift is detected
    v2_low_soc=0.25,                    # below this SoC only the classifier head is updated
    label_noise=0.0,                    # fraction of revealed labels that are wrong (robustness study)
    beta_sweep=[0.1, 0.25, 0.5, 1.0, 2.0],
    rho_sweep=[0.05, 0.10, 0.20, 0.50],          # label-availability sensitivity
    noise_sweep=[0.0, 0.10, 0.20, 0.30],         # label-noise robustness
    tau_min_grid=[0.3, 0.5, 0.7], tau_max_grid=[0.80, 0.90, 0.95, 0.99],
    ewr_sweep=[0.25, 1.0, 4.0, 16.0],            # nJ per parameter write (e_mac fixed) -> write/MAC cost ratio 1..64
    hidden_sweep=[(32, 16), (64, 32), (128, 64)],
    sweep_seeds=[0, 1, 2],
)
# Per-operation energy model (assumptions; see the note on energy values in the title cell). nJ per multiply-accumulate and per parameter write.
ENERGY = dict(e_mac_nj=0.25, e_write_nj=1.0)

if SMOKE:
    CFG.update(seeds=[0, 1], pre_epochs=10, beta_sweep=[0.25, 1.0], sweep_seeds=[0], rho_sweep=[0.1, 0.2],
               noise_sweep=[0.0, 0.2], tau_min_grid=[0.5], tau_max_grid=[0.9, 0.95], ewr_sweep=[1.0, 16.0],
               hidden_sweep=[(32, 16), (64, 32)])

MAIN_METHODS = ["Static", "Online-SGD", "TinyOL-Head", "RandomSkip", "Periodic-Full", "DriftTrig-Full", "GovRetrain",
                "CEGA-Tiny"]
# GovRetrain = energy-governed *buffer retraining* (our first design, v1); kept as a baseline.
# RandomSkip = control: skips labelled samples at random with the SAME update rate CEGA-Tiny used (same dataset/seed).
ABLATIONS = ["CEGA-Lite", "CEGA-noGate", "CEGA-noBurst", "CEGA-noSoC", "CEGA-noEntropy", "CEGA-HeadOnly"]
# CEGA-Lite = minimal configuration: gate + SoC-scaled threshold + head-only fallback (no replay, no entropy alarm).
ALL_METHODS = MAIN_METHODS + ABLATIONS
SWEEP_METHODS = ["Online-SGD", "Periodic-Full", "DriftTrig-Full", "CEGA-Tiny"]
print("Config ready. SMOKE =", SMOKE)

Config ready. SMOKE = False


In [2]:
CFG["seeds"] = list(range(10)); OUT = "cega_tiny_v5_10seed_results"; [os.makedirs(os.path.join(OUT, s), exist_ok=True) for s in ["figures", "tables", "raw"]]; import google.colab.files as _gf; _gf.download = lambda *a, **k: print("download skipped"); print("10 seeds:", CFG["seeds"], OUT)

10 seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] cega_tiny_v5_10seed_results


In [3]:
# ============================== 1. Data loading ==============================
UA = {"User-Agent": "Mozilla/5.0 (research script)"}

def download(url, dest, tries=3):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return dest
    last = None
    for t in range(tries):
        try:
            req = urllib.request.Request(url, headers=UA)
            with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
                shutil.copyfileobj(r, f)
            return dest
        except Exception as e:
            last = e; time.sleep(3 * (t + 1))
    raise RuntimeError(f"Download failed for {url}: {last}")

def _parse_gas_lines(text):
    X, y = [], []
    for line in text.splitlines():
        tok = line.strip().split()
        if not tok:
            continue
        y.append(int(tok[0].split(";")[0]) - 1)          # label format: "class;concentration"
        row = np.zeros(128, np.float32)
        for kv in tok[1:]:
            k, v = kv.split(":"); row[int(k) - 1] = float(v)
        X.append(row)
    return np.array(X), np.array(y)

def load_gas(seed):
    """UCI Gas Sensor Array Drift (Vergara et al., 2012). Pre-train on batches 1-2; stream batches 3-10."""
    p = download("https://archive.ics.uci.edu/static/public/224/gas+sensor+array+drift+dataset.zip",
                 os.path.join(DATA_DIR, "gas.zip"))
    Xs, ys, bs = [], [], []
    with zipfile.ZipFile(p) as z:
        names = {os.path.basename(n): n for n in z.namelist()}
        for b in range(1, 11):
            X, y = _parse_gas_lines(z.read(names[f"batch{b}.dat"]).decode())
            Xs.append(X); ys.append(y); bs.append(np.full(len(y), b))
    rng = np.random.default_rng(1000 + seed)
    Xp = np.vstack(Xs[:2]); yp = np.concatenate(ys[:2])
    Xstream, ystream, drift = [], [], []
    pos = 0
    for b in range(2, 10):
        perm = rng.permutation(len(ys[b]))                 # order inside a batch is unknown -> shuffle per seed
        Xstream.append(Xs[b][perm]); ystream.append(ys[b][perm])
        if b > 2: drift.append(pos)
        pos += len(perm)
    return dict(name="GasDrift", Xp=Xp, yp=yp, Xs=np.vstack(Xstream), ys=np.concatenate(ystream),
                drift=drift, C=6)

def load_har(seed):
    """UCI HAR (Anguita et al., 2013). Stream ordered by subject; pre-train on the first 6 subjects."""
    p = download("https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip",
                 os.path.join(DATA_DIR, "har.zip"))
    with zipfile.ZipFile(p) as z:
        inner = [n for n in z.namelist() if n.endswith("UCI HAR Dataset.zip")]
        zz = zipfile.ZipFile(io.BytesIO(z.read(inner[0]))) if inner else z
        def rd(part, kind):
            n = [m for m in zz.namelist() if m.endswith(f"{part}/{kind}_{part}.txt")][0]
            return pd.read_csv(io.BytesIO(zz.read(n)), sep=r"\s+", header=None).values
        X = np.vstack([rd("train", "X"), rd("test", "X")]).astype(np.float32)
        y = np.concatenate([rd("train", "y").ravel(), rd("test", "y").ravel()]) - 1
        s = np.concatenate([rd("train", "subject").ravel(), rd("test", "subject").ravel()])
    order = np.argsort(s, kind="stable")
    X, y, s = X[order], y[order], s[order]
    subj = np.unique(s); pre = np.isin(s, subj[:6])
    Xs, ys, ss = X[~pre], y[~pre], s[~pre]
    drift = [i for i in range(1, len(ss)) if ss[i] != ss[i - 1]]
    return dict(name="UCI-HAR", Xp=X[pre], yp=y[pre], Xs=Xs, ys=ys, drift=drift, C=6)

def load_elec(seed):
    """Electricity / Elec2 (Harries, 1999) from OpenML id 151. Chronological; pre-train on first 4000."""
    from sklearn.datasets import fetch_openml
    d = fetch_openml(data_id=151, as_frame=True, parser="auto", data_home=DATA_DIR)
    df = d.data.copy()
    df = df.drop(columns=[c for c in df.columns if c.lower() == "date"])
    X = df.apply(pd.to_numeric, errors="coerce").fillna(0).values.astype(np.float32)
    y = (d.target.astype(str).str.upper() == "UP").astype(int).values
    return dict(name="Elec2", Xp=X[:4000], yp=y[:4000], Xs=X[4000:], ys=y[4000:], drift=[], C=2)

def make_synthetic(name, seed, d=20, C=4, n_pre=1500, n_seg=1200, segs=5):
    """Smoke-test only: Gaussian classes whose means move at each segment. Never used for reported results."""
    rng = np.random.default_rng(seed + 7)
    mu = rng.normal(0, 2, (C, d))
    def seg(n, mu):
        y = rng.integers(0, C, n); return (mu[y] + rng.normal(0, 1, (n, d))).astype(np.float32), y
    Xp, yp = seg(n_pre, mu); Xs, ys, drift = [], [], []
    for k in range(segs):
        if k: mu = mu + rng.normal(0, 1.2, mu.shape); drift.append(k * n_seg)
        X, y = seg(n_seg, mu); Xs.append(X); ys.append(y)
    return dict(name=name, Xp=Xp, yp=yp, Xs=np.vstack(Xs), ys=np.concatenate(ys), drift=drift, C=C)

def load_covtype(seed):
    """Forest CoverType (Blackard & Dean, 1999), a standard real-world stream benchmark. First 60,000 records in
    original order; pre-train on the first 5,000."""
    from sklearn.datasets import fetch_covtype
    d = fetch_covtype(data_home=DATA_DIR)
    X = d.data[:60000].astype(np.float32); y = d.target[:60000].astype(int) - 1
    return dict(name="CoverType", Xp=X[:5000], yp=y[:5000], Xs=X[5000:], ys=y[5000:], drift=[], C=7)

LOADERS = {"GasDrift": load_gas, "UCI-HAR": load_har, "Elec2": load_elec, "CoverType": load_covtype}

def prepare(ds):
    """Standardise with pre-deployment statistics only (the device never sees future data)."""
    mu = ds["Xp"].mean(0); sd = ds["Xp"].std(0) + 1e-6
    f = lambda X: np.clip((X - mu) / sd, -8, 8).astype(np.float32)
    ds["Xp"], ds["Xs"] = f(ds["Xp"]), f(ds["Xs"])
    return ds

_DS_CACHE = {}
def get_dataset(name, seed):
    if (name, seed) not in _DS_CACHE:
        ds = make_synthetic(name, seed) if SMOKE else LOADERS[name](seed)
        _DS_CACHE[(name, seed)] = prepare(ds)
    return _DS_CACHE[(name, seed)]

DATASETS = ["GasDrift", "UCI-HAR", "Elec2", "CoverType"]
for n in DATASETS:
    d = get_dataset(n, 0)
    print(f"{n:9s} pre={len(d['yp']):6d} stream={len(d['ys']):6d} feat={d['Xs'].shape[1]:4d} "
          f"classes={d['C']} known drift points={len(d['drift'])}")

GasDrift  pre=  1689 stream= 12221 feat= 128 classes=6 known drift points=7
UCI-HAR   pre=  1934 stream=  8365 feat= 561 classes=6 known drift points=23
Elec2     pre=  4000 stream= 41312 feat=   7 classes=2 known drift points=0
CoverType pre=  5000 stream= 55000 feat=  54 classes=7 known drift points=0


In [4]:
# ============================== 2. Tiny MLP with exact op counting ==============================
class TinyMLP:
    def __init__(self, sizes, rng):
        self.sizes = list(sizes); self.L = len(sizes) - 1
        self.W = [rng.normal(0, np.sqrt(2 / i), (i, o)).astype(np.float32) for i, o in zip(sizes[:-1], sizes[1:])]
        self.b = [np.zeros(o, np.float32) for o in sizes[1:]]

    def copy(self):
        m = TinyMLP.__new__(TinyMLP); m.sizes = list(self.sizes); m.L = self.L
        m.W = [w.copy() for w in self.W]; m.b = [b.copy() for b in self.b]; return m

    def forward(self, X):
        acts, h = [X], X
        for l in range(self.L):
            z = h @ self.W[l] + self.b[l]
            h = np.maximum(z, 0) if l < self.L - 1 else z
            acts.append(h)
        return acts

    def proba(self, X):
        z = self.forward(X)[-1]; z = z - z.max(1, keepdims=True); e = np.exp(z)
        return e / e.sum(1, keepdims=True)

    def layer_macs(self):
        return [i * o for i, o in zip(self.sizes[:-1], self.sizes[1:])]

    def macs_fwd(self):
        return sum(self.layer_macs())

    def macs_train_sample(self, depth):
        lm = self.layer_macs(); top = lm[self.L - depth:]
        return self.macs_fwd() + sum(top) + sum(top[1:])   # forward + weight-grads + input-grads above the lowest trained layer

    def n_params(self, depth=None):
        depth = self.L if depth is None else depth
        return sum(self.W[l].size + self.b[l].size for l in range(self.L - depth, self.L))

    def grads(self, X, y, depth):
        acts = self.forward(X); n = len(X)
        z = acts[-1] - acts[-1].max(1, keepdims=True); p = np.exp(z); p /= p.sum(1, keepdims=True)
        loss = -np.log(p[np.arange(n), y] + 1e-9).mean()
        d = p; d[np.arange(n), y] -= 1; d /= n
        gW, gb = {}, {}
        for l in range(self.L - 1, self.L - depth - 1, -1):
            gW[l] = acts[l].T @ d; gb[l] = d.sum(0)
            if l > self.L - depth:
                d = (d @ self.W[l].T) * (acts[l] > 0)
        return gW, gb, loss

def loss_acc(model, X, y):
    p = model.proba(X)
    return float(-np.log(p[np.arange(len(y)), y] + 1e-9).mean()), float((p.argmax(1) == y).mean())

def sgd_train(model, X, y, depth, epochs, lr, rng, mom=0.9, batch=32, Xv=None, yv=None, early_stop=False, patience=1):
    """SGD with momentum on the top `depth` layers. Returns op counts for energy costing."""
    vel = {l: (np.zeros_like(model.W[l]), np.zeros_like(model.b[l])) for l in range(model.L - depth, model.L)}
    steps = samples = val_evals = 0; ep_run = 0
    best, bad, best_state = np.inf, 0, None
    for ep in range(epochs):
        perm = rng.permutation(len(X))
        for s in range(0, len(X), batch):
            idx = perm[s:s + batch]
            gW, gb, _ = model.grads(X[idx], y[idx], depth)
            gn = math.sqrt(sum(float((g ** 2).sum()) for g in gW.values()))
            sc = min(1.0, 5.0 / (gn + 1e-12))
            for l in gW:
                vW, vb = vel[l]
                vW *= mom; vW -= lr * sc * gW[l]; vb *= mom; vb -= lr * sc * gb[l]
                model.W[l] += vW; model.b[l] += vb
            steps += 1; samples += len(idx)
        ep_run += 1
        if early_stop and Xv is not None and len(yv):
            vl, _ = loss_acc(model, Xv, yv); val_evals += len(yv)
            if vl < best - 1e-4:
                best, bad = vl, 0
                best_state = [(model.W[l].copy(), model.b[l].copy()) for l in range(model.L - depth, model.L)]
            else:
                bad += 1
                if bad > patience: break
    if early_stop and best_state is not None:
        for k, l in enumerate(range(model.L - depth, model.L)):
            model.W[l], model.b[l] = best_state[k]
    return dict(steps=steps, samples=samples, val_evals=val_evals, epochs=ep_run)

def train_energy_mJ(model, depth, ops):
    e = ops["samples"] * model.macs_train_sample(depth) * ENERGY["e_mac_nj"]
    e += ops["val_evals"] * model.macs_fwd() * ENERGY["e_mac_nj"]
    e += ops["steps"] * model.n_params(depth) * ENERGY["e_write_nj"]
    return e * 1e-6   # nJ -> mJ

def planned_energy_mJ(model, depth, n, epochs, n_val=0):
    steps = epochs * math.ceil(n / CFG["batch"])
    return train_energy_mJ(model, depth, dict(samples=epochs * n, steps=steps, val_evals=epochs * n_val))

In [5]:
# ============================== 3. Drift detection ==============================
class PageHinkley:
    """Page-Hinkley test for an increase in the mean of a signal."""
    def __init__(self, delta, lam, min_n):
        self.delta, self.lam, self.min_n = delta, lam, min_n; self.reset()
    def reset(self):
        self.n, self.mean, self.cum, self.min = 0, 0.0, 0.0, 0.0
    def update(self, x):
        self.n += 1; self.mean += (x - self.mean) / self.n
        self.cum += x - self.mean - self.delta; self.min = min(self.min, self.cum)
        return self.n >= self.min_n and (self.cum - self.min) > self.lam

def norm_entropy(P):
    return float((-(P * np.log(P + 1e-12)).sum(1) / np.log(P.shape[1])).mean())

In [6]:
# ============================== 4. Streaming simulator (all methods) ==============================
PRETRAINED = {}

def pretrained_model(ds, seed):
    key = (ds["name"], seed, tuple(CFG["hidden"]))
    if key not in PRETRAINED:
        rng = np.random.default_rng(seed)
        n = len(ds["yp"]); perm = rng.permutation(n); nv = int(0.15 * n)
        tr, va = perm[nv:], perm[:nv]
        m = TinyMLP([ds["Xs"].shape[1], *CFG["hidden"], ds["C"]], rng)
        sgd_train(m, ds["Xp"][tr], ds["yp"][tr], m.L, CFG["pre_epochs"], CFG["pre_lr"], rng, batch=64)
        _, acc = loss_acc(m, ds["Xp"][va], ds["yp"][va])
        P = m.proba(ds["Xp"][va])
        PRETRAINED[key] = (m, 1 - acc, norm_entropy(P))
    return PRETRAINED[key]

def recovery_windows(win_acc, drift_samples, W, cap=20, pre=5):
    """Windows needed after a known drift point for the 3-window mean accuracy to regain 95% of the pre-drift level."""
    wa = np.asarray(win_acc); out = []
    dws = sorted(set(d // W for d in drift_samples))
    for i, dw in enumerate(dws):
        if dw < pre or dw >= len(wa) - 3: continue
        nxt = dws[i + 1] if i + 1 < len(dws) else len(wa)
        ref = wa[dw - pre:dw].mean(); r = cap
        for k in range(dw, min(dw + cap, nxt, len(wa) - 2)):
            if wa[k:k + 3].mean() >= 0.95 * ref: r = k - dw; break
        out.append(r)
    return float(np.mean(out)) if out else np.nan

def run_stream(ds, method, seed, beta=None, keep_trace=False, keep=None):
    beta = CFG["beta"] if beta is None else beta
    base, pre_err, pre_ent = pretrained_model(ds, seed)
    model = base.copy(); rng = np.random.default_rng(10_000 + seed)
    Xs, ys, W, B = ds["Xs"], ds["ys"], CFG["window"], CFG["buffer"]
    nwin = len(ys) // W; n = nwin * W
    lab = np.random.default_rng(20_000 + seed).random(n) < CFG["p_label"]   # same label mask for every method
    # labels the device observes (optionally corrupted); evaluation always uses the true labels ys
    yo = ys.copy()
    if CFG.get("label_noise", 0.0) > 0:
        nr = np.random.default_rng(30_000 + seed)
        idx = np.where(lab & (nr.random(n) < CFG["label_noise"]))[0]
        yo[idx] = (ys[idx] + nr.integers(1, ds["C"], len(idx))) % ds["C"]
    rs_rng = np.random.default_rng(40_000 + seed)
    buf = deque(maxlen=B)

    L = model.L
    E_full = planned_energy_mJ(model, L, B, CFG["max_epochs"])
    cap = CFG["cap_mult"] * E_full; batt = CFG["soc0"] * cap
    E_inf_win = W * model.macs_fwd() * ENERGY["e_mac_nj"] * 1e-6
    harvest = E_inf_win + beta * E_full / 20.0

    # method switches
    is_v2 = method.startswith("CEGA")          # proposed CEGA-Tiny and its ablations
    is_cega = method == "GovRetrain"           # v1 design: energy-governed buffer retraining
    v2_ent = method not in ("CEGA-noEntropy", "CEGA-Lite"); v2_gate = method != "CEGA-noGate"
    v2_burst = method not in ("CEGA-noBurst", "CEGA-Lite"); v2_soc = method != "CEGA-noSoC"; v2_head = method == "CEGA-HeadOnly"
    acts = dict(skip=0, head=0, full=0, burst=0)
    n_lab = n_upd = 0
    boost = 0
    use_ent = calib = use_soc = use_es = True; depths = [1, 2, 3]
    g = dict(CFG["g_prior"])

    ph_err = PageHinkley(**CFG["ph_err"]); ph_ent = PageHinkley(**CFG["ph_ent"])
    preds = np.empty(n, int); win_acc, soc_tr, events = [], [], []
    hist_err, hist_n, hist_ent = [], [], []          # per-window labelled errors / counts / entropy since last reset
    E_inf = E_ad = 0.0; n_ad = viol = 0; deficit = 0.0; min_soc = 1.0
    pending = cool = 0; depth_count = {1: 0, 2: 0, 3: 0}

    def buf_xy():
        idx = np.fromiter(buf, int); return Xs[idx], yo[idx]

    for w in range(nwin):
        sl = slice(w * W, (w + 1) * W); Xw, yw, ywo = Xs[sl], ys[sl], yo[sl]
        P = model.proba(Xw); pw = P.argmax(1); preds[sl] = pw
        win_acc.append(float((pw == yw).mean()))
        batt = min(cap, batt + harvest) - E_inf_win; E_inf += E_inf_win
        li = np.where(lab[sl])[0]
        for i in li: buf.append(w * W + i)
        errs = (pw[li] != ywo[li]).astype(float); H = norm_entropy(P); n_lab += len(li)
        hist_err.append(errs.sum()); hist_n.append(len(li)); hist_ent.append(H)
        E_step = 0.0; cool = max(0, cool - 1)

        if method in ("Online-SGD", "TinyOL-Head") and len(li):
            d = L if method == "Online-SGD" else 1
            ops = sgd_train(model, Xw[li], ywo[li], d, 1, CFG["lr_online"], rng, CFG["momentum"], CFG["batch"])
            E_step = train_energy_mJ(model, d, ops); n_ad += 1; n_upd += len(li)

        elif method == "RandomSkip" and len(li):
            # control: keeps each labelled sample with probability `keep` (matched to CEGA-Tiny's update rate)
            sel = li[rs_rng.random(len(li)) < keep]
            if len(sel):
                ops = sgd_train(model, Xw[sel], ywo[sel], L, 1, CFG["lr_online"], rng, CFG["momentum"], CFG["batch"])
                E_step = train_energy_mJ(model, L, ops); n_ad += 1; n_upd += len(sel)

        elif method == "Periodic-Full":
            if (w + 1) % CFG["period"] == 0 and len(buf) >= CFG["min_buffer"]:
                Xb, yb = buf_xy()
                ops = sgd_train(model, Xb, yb, L, CFG["adapt_epochs"], CFG["lr_adapt"], rng, CFG["momentum"], CFG["batch"])
                E_step = train_energy_mJ(model, L, ops); n_ad += 1; events.append((w, L))

        elif method == "DriftTrig-Full":
            fired = any([ph_err.update(e) for e in errs])
            if fired and cool == 0 and len(buf) >= CFG["min_buffer"]:
                Xb, yb = buf_xy()
                ops = sgd_train(model, Xb, yb, L, CFG["adapt_epochs"], CFG["lr_adapt"], rng, CFG["momentum"], CFG["batch"])
                E_step = train_energy_mJ(model, L, ops); n_ad += 1; events.append((w, L))
                ph_err.reset(); cool = CFG["cooldown"]

        elif is_v2:
            fired_err = any([ph_err.update(e) for e in errs])
            fired_ent = ph_ent.update(H) if v2_ent else False
            if (fired_err or fired_ent) and cool == 0:
                # entropy alarm (label-free) -> ungated boost only; error alarm (label evidence) -> boost + replay burst
                boost = CFG["v2_boost"]; cool = CFG["cooldown"]; ph_err.reset(); ph_ent.reset()
                if fired_err and v2_burst and len(buf) >= CFG["min_buffer"]:
                    Ea = planned_energy_mJ(model, L, len(buf), CFG["v2_burst_epochs"])
                    if batt - Ea >= CFG["reserve"] * cap:
                        Xb, yb = buf_xy()
                        ops = sgd_train(model, Xb, yb, L, CFG["v2_burst_epochs"], CFG["lr_adapt"], rng, CFG["momentum"], CFG["batch"])
                        E_step += train_energy_mJ(model, L, ops); n_ad += 1; acts["burst"] += 1; events.append((w, L))
            if len(li):
                soc = max(batt - E_step, 0.0) / cap
                if boost > 0 or not v2_gate:
                    tau = 1.01
                else:
                    tau = CFG["v2_tau_min"] + (CFG["v2_tau_max"] - CFG["v2_tau_min"]) * (min(soc, 1.0) if v2_soc else 0.5)
                boost = max(0, boost - 1)
                sel = li[P[li, ywo[li]] < tau]
                if not len(sel): acts["skip"] += 1
                if len(sel):
                    d = 1 if (v2_head or (v2_soc and soc < CFG["v2_low_soc"])) else L
                    Ea = planned_energy_mJ(model, d, len(sel), 1)
                    if batt - E_step - Ea >= 0:
                        ops = sgd_train(model, Xw[sel], ywo[sel], d, 1, CFG["lr_online"], rng, CFG["momentum"], CFG["batch"])
                        E_step += train_energy_mJ(model, d, ops); n_ad += 1; acts["head" if d == 1 else "full"] += 1
                        n_upd += len(sel)
                    else:
                        acts["skip"] += 1

        elif is_cega:
            fired = any([ph_err.update(e) for e in errs])
            if use_ent: fired = ph_ent.update(H) or fired
            if fired and cool == 0: pending = CFG["pending_max"]
            if pending > 0 and len(buf) >= CFG["min_buffer"]:
                r = CFG["recent"]
                er_n = sum(hist_n[-r:]); er = sum(hist_err[-r:]) / max(er_n, 1)
                ref_n = sum(hist_n[:-r]); ref = sum(hist_err[:-r]) / ref_n if ref_n >= 20 else pre_err
                sev = np.clip((er - ref) / max(ref, 0.05), 0, 2) / 2 if er_n else 0.0
                if use_ent:
                    href = np.mean(hist_ent[:-r]) if len(hist_ent) > r + 2 else pre_ent
                    sev = max(sev, np.clip((np.mean(hist_ent[-r:]) - href) / max(href, 0.05), 0, 2) / 2)
                soc = max(batt / cap, 0.05)
                lam_t = CFG["lambda0"] / (soc if use_soc else 0.5)
                n_tr = int(0.8 * len(buf)); n_va = len(buf) - n_tr
                best_d, best_u = None, 0.0
                for d in depths:
                    Ea = planned_energy_mJ(model, d, n_tr, CFG["max_epochs"] if use_es else CFG["adapt_epochs"], n_va if use_es else 0)
                    u = sev * g[d] - lam_t * (Ea / E_full) * CFG["g_scale"]
                    if batt - Ea >= CFG["reserve"] * cap and u > best_u:
                        best_d, best_u = d, u
                if best_d is not None:
                    Xb, yb = buf_xy()
                    Xt, yt, Xv, yv = Xb[:n_tr], yb[:n_tr], Xb[n_tr:], yb[n_tr:]   # holdout = most recent 20%
                    _, a0 = loss_acc(model, Xv, yv); E_step += n_va * model.macs_fwd() * ENERGY["e_mac_nj"] * 1e-6
                    ops = sgd_train(model, Xt, yt, best_d, CFG["max_epochs"] if use_es else CFG["adapt_epochs"],
                                    CFG["lr_adapt"], rng, CFG["momentum"], CFG["batch"], Xv, yv, early_stop=use_es)
                    _, a1 = loss_acc(model, Xv, yv); E_step += n_va * model.macs_fwd() * ENERGY["e_mac_nj"] * 1e-6
                    E_step += train_energy_mJ(model, best_d, ops)
                    if calib:
                        g[best_d] = (1 - CFG["g_ema"]) * g[best_d] + CFG["g_ema"] * max(a1 - a0, 0.005)
                    n_ad += 1; depth_count[best_d] += 1; events.append((w, best_d))
                    pending = 0; cool = CFG["cooldown"]; ph_err.reset(); ph_ent.reset()
                    hist_err, hist_n, hist_ent = hist_err[-1:], hist_n[-1:], hist_ent[-1:]
                else:
                    pending -= 1

        if E_step > 0:
            E_ad += E_step; batt -= E_step
            if batt < 0:
                viol += 1; deficit += -batt; batt = 0.0
        min_soc = min(min_soc, batt / cap); soc_tr.append(batt / cap)

    y_true = ys[:n]
    res = dict(dataset=ds["name"], method=method, seed=seed, beta=beta,
               acc=float((preds == y_true).mean()),
               macro_f1=float(f1_score(y_true, preds, average="macro")),
               E_inf_mJ=E_inf, E_adapt_mJ=E_ad, E_total_mJ=E_inf + E_ad, n_adapt=n_ad,
               violations=viol, deficit_mJ=deficit, min_soc=min_soc,
               recovery_win=recovery_windows(win_acc, ds["drift"], W),
               d1=depth_count[1], d2=depth_count[2], d3=depth_count[3], E_full_mJ=E_full,
               skip=acts["skip"], head=acts["head"], full=acts["full"], burst=acts["burst"],
               n_labelled=n_lab, n_updated=n_upd, frac_updated=(n_upd / n_lab if n_lab else np.nan),
               p_label=CFG["p_label"], label_noise=CFG.get("label_noise", 0.0), tau_min=CFG["v2_tau_min"],
               tau_max=CFG["v2_tau_max"], e_wr_nj=ENERGY["e_write_nj"], hidden=str(tuple(CFG["hidden"])))
    if keep_trace:
        res["_trace"] = dict(win_acc=win_acc, soc=soc_tr, events=events)
    return res

In [7]:
# ============================== 5. Helpers for experiments ==============================
from contextlib import contextmanager

@contextmanager
def override(cfg=None, energy=None):
    """Temporarily change CFG / ENERGY entries (restored afterwards)."""
    cfg, energy = cfg or {}, energy or {}
    old_c = {k: CFG[k] for k in cfg}; old_e = {k: ENERGY[k] for k in energy}
    CFG.update(cfg); ENERGY.update(energy)
    try:
        yield
    finally:
        CFG.update(old_c); ENERGY.update(old_e)

def run_methods(ds, seed, methods, keep_trace=False):
    """Runs `methods` on one dataset/seed. RandomSkip is matched to CEGA-Tiny's update rate on the same run."""
    rows, out_traces, frac = [], {}, None
    order = [m for m in methods if m != "RandomSkip"]
    need_rs = "RandomSkip" in methods
    if need_rs and "CEGA-Tiny" not in order:
        frac = run_stream(ds, "CEGA-Tiny", seed)["frac_updated"]
    for m in order + (["RandomSkip"] if need_rs else []):
        if m == "RandomSkip" and frac is None:
            frac = [r for r in rows if r["method"] == "CEGA-Tiny"][0]["frac_updated"]
        r = run_stream(ds, m, seed, keep_trace=keep_trace, keep=(frac if m == "RandomSkip" else None))
        tr = r.pop("_trace", None)
        if tr is not None: out_traces[m] = tr
        rows.append(r)
    return rows, out_traces

In [8]:
# ============================== B0. Part B extensions (delay, solar harvest, int8, matched baselines, new streams)

CFG.update(label_delay=0, harvest_profile="const", solar_period=24, quant_bits=0)

def harvest_w(harvest, E_inf_win, w):
    """Constant harvest (v3) or a day/night solar profile with the same mean surplus."""
    if CFG.get("harvest_profile", "const") == "const":
        return harvest
    P = CFG["solar_period"]; surplus = harvest - E_inf_win
    return E_inf_win + surplus * math.pi * max(0.0, math.sin(2 * math.pi * (w % P) / P))

def _fq(a, bits):
    s = float(np.abs(a).max()) / (2 ** (bits - 1) - 1)
    return a if s == 0 else (np.clip(np.round(a / s), -(2 ** (bits - 1) - 1), 2 ** (bits - 1) - 1) * s).astype(np.float32)

def proba_q(model, X, bits=8):
    """Fake-quantised inference: per-tensor symmetric int8 weights and activations (float32 master weights)."""
    h = _fq(X, bits)
    for l in range(model.L):
        z = h @ _fq(model.W[l], bits) + model.b[l]
        h = _fq(np.maximum(z, 0), bits) if l < model.L - 1 else z
    z = h - h.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

def run_stream4(ds, method, seed, beta=None, keep_trace=False, keep=None):
    """v3 run_stream + Part B options (CFG keys): label_delay (windows; the gate uses the softmax stored at inference time), harvest_profile (const|solar),
    quant_bits (0|8, fake-quantised inference), and matched baselines SB-Matched / Entropy-Matched.
    With label_delay=0, harvest_profile=const, quant_bits=0 it reproduces run_stream exactly."""
    beta = CFG["beta"] if beta is None else beta
    base, pre_err, pre_ent = pretrained_model(ds, seed)
    model = base.copy(); rng = np.random.default_rng(10_000 + seed)
    Xs, ys, W, B = ds["Xs"], ds["ys"], CFG["window"], CFG["buffer"]
    nwin = len(ys) // W; n = nwin * W
    lab = np.random.default_rng(20_000 + seed).random(n) < CFG["p_label"]   # same label mask for every method
    # labels the device observes (optionally corrupted); evaluation always uses the true labels ys
    yo = ys.copy()
    if CFG.get("label_noise", 0.0) > 0:
        nr = np.random.default_rng(30_000 + seed)
        idx = np.where(lab & (nr.random(n) < CFG["label_noise"]))[0]
        yo[idx] = (ys[idx] + nr.integers(1, ds["C"], len(idx))) % ds["C"]
    rs_rng = np.random.default_rng(40_000 + seed)
    mt_rng = np.random.default_rng(50_000 + seed)
    D = int(CFG.get("label_delay", 0)); pending_lab = deque()
    Pstore = np.zeros((len(ds["ys"]), ds["C"]), np.float32) if D > 0 else None
    qb = int(CFG.get("quant_bits", 0)); proba = (lambda X: proba_q(model, X, qb)) if qb else (lambda X: model.proba(X))
    buf = deque(maxlen=B)

    L = model.L
    E_full = planned_energy_mJ(model, L, B, CFG["max_epochs"])
    cap = CFG["cap_mult"] * E_full; batt = CFG["soc0"] * cap
    E_inf_win = W * model.macs_fwd() * ENERGY["e_mac_nj"] * 1e-6
    harvest = E_inf_win + beta * E_full / 20.0

    # method switches
    is_v2 = method.startswith("CEGA")          # proposed CEGA-Tiny and its ablations
    is_cega = method == "GovRetrain"           # v1 design: energy-governed buffer retraining
    v2_ent = method not in ("CEGA-noEntropy", "CEGA-Lite"); v2_gate = method != "CEGA-noGate"
    v2_burst = method not in ("CEGA-noBurst", "CEGA-Lite"); v2_soc = method != "CEGA-noSoC"; v2_head = method == "CEGA-HeadOnly"
    acts = dict(skip=0, head=0, full=0, burst=0)
    n_lab = n_upd = 0
    boost = 0
    use_ent = calib = use_soc = use_es = True; depths = [1, 2, 3]
    g = dict(CFG["g_prior"])

    ph_err = PageHinkley(**CFG["ph_err"]); ph_ent = PageHinkley(**CFG["ph_ent"])
    preds = np.empty(n, int); win_acc, soc_tr, events = [], [], []
    hist_err, hist_n, hist_ent = [], [], []          # per-window labelled errors / counts / entropy since last reset
    E_inf = E_ad = 0.0; n_ad = viol = 0; deficit = 0.0; min_soc = 1.0
    pending = cool = 0; depth_count = {1: 0, 2: 0, 3: 0}

    def buf_xy():
        idx = np.fromiter(buf, int); return Xs[idx], yo[idx]

    for w in range(nwin):
        sl = slice(w * W, (w + 1) * W); Xw, yw, ywo = Xs[sl], ys[sl], yo[sl]
        P = proba(Xw); pw = P.argmax(1); preds[sl] = pw
        if D > 0: Pstore[sl] = P            # keep the inference softmax until the delayed label arrives
        win_acc.append(float((pw == yw).mean()))
        batt = min(cap, batt + harvest_w(harvest, E_inf_win, w)) - E_inf_win; E_inf += E_inf_win
        li_new = w * W + np.where(lab[sl])[0]
        if D == 0:
            gidx = li_new
        else:
            pending_lab.append((w + D, li_new)); gidx = np.empty(0, int)
            while pending_lab and pending_lab[0][0] <= w: gidx = np.concatenate([gidx, pending_lab.popleft()[1]])
        for i in gidx: buf.append(int(i))
        Xl, yl = Xs[gidx], yo[gidx]
        errs = (preds[gidx] != yl).astype(float); H = norm_entropy(P); n_lab += len(gidx)
        _Pl = [None]
        def Pl():
            if _Pl[0] is None:
                _Pl[0] = P[gidx - w * W] if D == 0 else Pstore[gidx]   # probability computed at inference time
            return _Pl[0]
        li = gidx
        hist_err.append(errs.sum()); hist_n.append(len(li)); hist_ent.append(H)
        E_step = 0.0; cool = max(0, cool - 1)

        if method in ("Online-SGD", "TinyOL-Head") and len(li):
            d = L if method == "Online-SGD" else 1
            ops = sgd_train(model, Xl, yl, d, 1, CFG["lr_online"], rng, CFG["momentum"], CFG["batch"])
            E_step = train_energy_mJ(model, d, ops); n_ad += 1; n_upd += len(li)

        elif method == "RandomSkip" and len(li):
            # control: keeps each labelled sample with probability `keep` (matched to CEGA-Tiny's update rate)
            sel = np.where(rs_rng.random(len(li)) < keep)[0]
            if len(sel):
                ops = sgd_train(model, Xl[sel], yl[sel], L, 1, CFG["lr_online"], rng, CFG["momentum"], CFG["batch"])
                E_step = train_energy_mJ(model, L, ops); n_ad += 1; n_upd += len(sel)

        elif method in ("SB-Matched", "Entropy-Matched") and len(li):
            # matched-rate selection: same expected update count as CEGA-Tiny; SB = largest loss (Jiang et al.),
            # Entropy = most uncertain prediction (label-free uncertainty sampling). All layers, one SGD epoch.
            k = int(np.floor(keep * len(li) + mt_rng.random()))
            if k > 0:
                Pv = Pl()
                score = -np.log(Pv[np.arange(len(li)), yl] + 1e-12) if method == "SB-Matched" else \
                        -(Pv * np.log(Pv + 1e-12)).sum(1)
                sel = np.argsort(-score, kind="stable")[:k]
                ops = sgd_train(model, Xl[sel], yl[sel], L, 1, CFG["lr_online"], rng, CFG["momentum"], CFG["batch"])
                E_step += train_energy_mJ(model, L, ops); n_ad += 1; n_upd += len(sel)

        elif method == "Periodic-Full":
            if (w + 1) % CFG["period"] == 0 and len(buf) >= CFG["min_buffer"]:
                Xb, yb = buf_xy()
                ops = sgd_train(model, Xb, yb, L, CFG["adapt_epochs"], CFG["lr_adapt"], rng, CFG["momentum"], CFG["batch"])
                E_step = train_energy_mJ(model, L, ops); n_ad += 1; events.append((w, L))

        elif method == "DriftTrig-Full":
            fired = any([ph_err.update(e) for e in errs])
            if fired and cool == 0 and len(buf) >= CFG["min_buffer"]:
                Xb, yb = buf_xy()
                ops = sgd_train(model, Xb, yb, L, CFG["adapt_epochs"], CFG["lr_adapt"], rng, CFG["momentum"], CFG["batch"])
                E_step = train_energy_mJ(model, L, ops); n_ad += 1; events.append((w, L))
                ph_err.reset(); cool = CFG["cooldown"]

        elif is_v2:
            fired_err = any([ph_err.update(e) for e in errs])
            fired_ent = ph_ent.update(H) if v2_ent else False
            if (fired_err or fired_ent) and cool == 0:
                # entropy alarm (label-free) -> ungated boost only; error alarm (label evidence) -> boost + replay burst
                boost = CFG["v2_boost"]; cool = CFG["cooldown"]; ph_err.reset(); ph_ent.reset()
                if fired_err and v2_burst and len(buf) >= CFG["min_buffer"]:
                    Ea = planned_energy_mJ(model, L, len(buf), CFG["v2_burst_epochs"])
                    if batt - Ea >= CFG["reserve"] * cap:
                        Xb, yb = buf_xy()
                        ops = sgd_train(model, Xb, yb, L, CFG["v2_burst_epochs"], CFG["lr_adapt"], rng, CFG["momentum"], CFG["batch"])
                        E_step += train_energy_mJ(model, L, ops); n_ad += 1; acts["burst"] += 1; events.append((w, L))
            if len(li):
                soc = max(batt - E_step, 0.0) / cap
                if boost > 0 or not v2_gate:
                    tau = 1.01
                else:
                    tau = CFG["v2_tau_min"] + (CFG["v2_tau_max"] - CFG["v2_tau_min"]) * (min(soc, 1.0) if v2_soc else 0.5)
                boost = max(0, boost - 1)
                sel = np.where(Pl()[np.arange(len(li)), yl] < tau)[0]
                if not len(sel): acts["skip"] += 1
                if len(sel):
                    d = 1 if (v2_head or (v2_soc and soc < CFG["v2_low_soc"])) else L
                    Ea = planned_energy_mJ(model, d, len(sel), 1)
                    if batt - E_step - Ea >= 0:
                        ops = sgd_train(model, Xl[sel], yl[sel], d, 1, CFG["lr_online"], rng, CFG["momentum"], CFG["batch"])
                        E_step += train_energy_mJ(model, d, ops); n_ad += 1; acts["head" if d == 1 else "full"] += 1
                        n_upd += len(sel)
                    else:
                        acts["skip"] += 1

        elif is_cega:
            fired = any([ph_err.update(e) for e in errs])
            if use_ent: fired = ph_ent.update(H) or fired
            if fired and cool == 0: pending = CFG["pending_max"]
            if pending > 0 and len(buf) >= CFG["min_buffer"]:
                r = CFG["recent"]
                er_n = sum(hist_n[-r:]); er = sum(hist_err[-r:]) / max(er_n, 1)
                ref_n = sum(hist_n[:-r]); ref = sum(hist_err[:-r]) / ref_n if ref_n >= 20 else pre_err
                sev = np.clip((er - ref) / max(ref, 0.05), 0, 2) / 2 if er_n else 0.0
                if use_ent:
                    href = np.mean(hist_ent[:-r]) if len(hist_ent) > r + 2 else pre_ent
                    sev = max(sev, np.clip((np.mean(hist_ent[-r:]) - href) / max(href, 0.05), 0, 2) / 2)
                soc = max(batt / cap, 0.05)
                lam_t = CFG["lambda0"] / (soc if use_soc else 0.5)
                n_tr = int(0.8 * len(buf)); n_va = len(buf) - n_tr
                best_d, best_u = None, 0.0
                for d in depths:
                    Ea = planned_energy_mJ(model, d, n_tr, CFG["max_epochs"] if use_es else CFG["adapt_epochs"], n_va if use_es else 0)
                    u = sev * g[d] - lam_t * (Ea / E_full) * CFG["g_scale"]
                    if batt - Ea >= CFG["reserve"] * cap and u > best_u:
                        best_d, best_u = d, u
                if best_d is not None:
                    Xb, yb = buf_xy()
                    Xt, yt, Xv, yv = Xb[:n_tr], yb[:n_tr], Xb[n_tr:], yb[n_tr:]   # holdout = most recent 20%
                    _, a0 = loss_acc(model, Xv, yv); E_step += n_va * model.macs_fwd() * ENERGY["e_mac_nj"] * 1e-6
                    ops = sgd_train(model, Xt, yt, best_d, CFG["max_epochs"] if use_es else CFG["adapt_epochs"],
                                    CFG["lr_adapt"], rng, CFG["momentum"], CFG["batch"], Xv, yv, early_stop=use_es)
                    _, a1 = loss_acc(model, Xv, yv); E_step += n_va * model.macs_fwd() * ENERGY["e_mac_nj"] * 1e-6
                    E_step += train_energy_mJ(model, best_d, ops)
                    if calib:
                        g[best_d] = (1 - CFG["g_ema"]) * g[best_d] + CFG["g_ema"] * max(a1 - a0, 0.005)
                    n_ad += 1; depth_count[best_d] += 1; events.append((w, best_d))
                    pending = 0; cool = CFG["cooldown"]; ph_err.reset(); ph_ent.reset()
                    hist_err, hist_n, hist_ent = hist_err[-1:], hist_n[-1:], hist_ent[-1:]
                else:
                    pending -= 1

        if E_step > 0:
            E_ad += E_step; batt -= E_step
            if batt < 0:
                viol += 1; deficit += -batt; batt = 0.0
        min_soc = min(min_soc, batt / cap); soc_tr.append(batt / cap)

    y_true = ys[:n]
    res = dict(dataset=ds["name"], method=method, seed=seed, beta=beta,
               acc=float((preds == y_true).mean()),
               macro_f1=float(f1_score(y_true, preds, average="macro")),
               E_inf_mJ=E_inf, E_adapt_mJ=E_ad, E_total_mJ=E_inf + E_ad, n_adapt=n_ad,
               violations=viol, deficit_mJ=deficit, min_soc=min_soc,
               recovery_win=recovery_windows(win_acc, ds["drift"], W),
               d1=depth_count[1], d2=depth_count[2], d3=depth_count[3], E_full_mJ=E_full,
               skip=acts["skip"], head=acts["head"], full=acts["full"], burst=acts["burst"],
               n_labelled=n_lab, n_updated=n_upd, frac_updated=(n_upd / n_lab if n_lab else np.nan),
               p_label=CFG["p_label"], label_noise=CFG.get("label_noise", 0.0), tau_min=CFG["v2_tau_min"],
               tau_max=CFG["v2_tau_max"], e_wr_nj=ENERGY["e_write_nj"], hidden=str(tuple(CFG["hidden"])),
               label_delay=D, harvest_profile=CFG.get("harvest_profile", "const"), quant_bits=qb)
    if keep_trace:
        res["_trace"] = dict(win_acc=win_acc, soc=soc_tr, events=events)
    return res



MATCHED = ("RandomSkip", "SB-Matched", "Entropy-Matched")

def run_methods4(ds, seed, methods, keep_trace=False):
    """run_methods for v4: matched-rate controls use CEGA-Tiny's update fraction on the same run."""
    rows, out_traces, frac = [], {}, None
    order = [m for m in methods if m not in MATCHED]; need = [m for m in methods if m in MATCHED]
    if need and "CEGA-Tiny" not in order:
        frac = run_stream4(ds, "CEGA-Tiny", seed)["frac_updated"]
    for m in order + need:
        if m in MATCHED and frac is None:
            frac = [r for r in rows if r["method"] == "CEGA-Tiny"][0]["frac_updated"]
        r = run_stream4(ds, m, seed, keep_trace=keep_trace, keep=(frac if m in MATCHED else None))
        tr = r.pop("_trace", None)
        if tr is not None: out_traces[m] = tr
        rows.append(r)
    return rows, out_traces

# ------------------------------------------------------------------ new datasets (Part B)
WEATHER_URL = "https://raw.githubusercontent.com/vlosing/driftDatasets/master/realWorld/weather/"

def load_sea(seed, noise=0.10):
    """SEA concepts (Street & Kim, 2001): x in [0,10]^3, y = 1[x1 + x2 <= theta], 10% class noise.
    Pre-deployment: 3,000 samples of theta = 8. Stream: 12,000 more of theta = 8, then 15,000 each of 9, 7, 9.5."""
    rng = np.random.default_rng(2000 + seed)
    def block(n, th):
        X = rng.uniform(0, 10, (n, 3)).astype(np.float32); y = (X[:, 0] + X[:, 1] <= th).astype(int)
        flip = rng.random(n) < noise; y[flip] = 1 - y[flip]; return X, y
    Xp, yp = block(3000, 8.0); parts = [block(12000, 8.0), block(15000, 9.0), block(15000, 7.0), block(15000, 9.5)]
    Xs = np.vstack([p[0] for p in parts]); ys = np.concatenate([p[1] for p in parts])
    return dict(name="SEA", Xp=Xp, yp=yp, Xs=Xs, ys=ys, drift=[12000, 27000, 42000], C=2)

def load_weather(seed):
    """NOAA weather, Bellevue, Nebraska (Elwell & Polikar, 2011): 18,159 daily records, 8 sensor features,
    rain / no rain. Chronological; pre-train on the first 1,800 days (about five years)."""
    fd = download(WEATHER_URL + "NEweather_data.csv", os.path.join(DATA_DIR, "NEweather_data.csv"))
    fc = download(WEATHER_URL + "NEweather_class.csv", os.path.join(DATA_DIR, "NEweather_class.csv"))
    X = pd.read_csv(fd, header=None).values.astype(np.float32); y = pd.read_csv(fc, header=None).values.ravel().astype(int)
    return dict(name="Weather", Xp=X[:1800], yp=y[:1800], Xs=X[1800:], ys=y[1800:], drift=[], C=2)

LOADERS.update({"SEA": load_sea, "Weather": load_weather})
DATASETS6 = ["GasDrift", "UCI-HAR", "Elec2", "CoverType", "SEA", "Weather"]


In [9]:
# ============================== B1. Regression check: v4 simulator == v3 simulator on the original four streams
keys = ["acc", "macro_f1", "E_adapt_mJ", "violations", "n_updated"]; bad = 0
for n in DATASETS:
    ds = get_dataset(n, 0)
    r3, _ = run_methods(ds, 0, ALL_METHODS); r4, _ = run_methods4(ds, 0, ALL_METHODS)
    for a, b in zip(r3, r4):
        bad += sum(a[k] != b[k] for k in keys)
print("REGRESSION CHECK:", "PASS (v4 reproduces v3 exactly)" if bad == 0 else f"FAIL ({bad} mismatches)")


REGRESSION CHECK: PASS (v4 reproduces v3 exactly)


In [10]:
# ============================== B2. Part B experiments
OUT_B = OUT
D6 = DATASETS6
M_MAIN = MAIN_METHODS + ["CEGA-Lite", "SB-Matched", "Entropy-Matched"]

def sweep4(name, values, methods, make_override, seeds=None, datasets=D6):
    t0 = time.time(); out = []
    for n in datasets:
        for seed in (seeds or CFG["sweep_seeds"]):
            ds = get_dataset(n, seed)
            for v in values:
                c, e = make_override(v)
                with override(c, e):
                    r, _ = run_methods4(ds, seed, methods)
                for x in r: x.update(sweep=name, value=str(v))
                out += r
    df = pd.DataFrame(out); df.to_csv(f"{OUT}/raw/sweep_{name}.csv", index=False)
    print(f"sweep {name:8s}: {len(df)} runs in {time.time() - t0:.0f}s", flush=True); return df

# ---------------- B1: main comparison on six streams (adds SEA, Weather, SB-Matched, Entropy-Matched)
t0 = time.time(); rows = []
for n in D6:
    for seed in CFG["seeds"]:
        r, _ = run_methods4(get_dataset(n, seed), seed, M_MAIN); rows += r
    print(f"main {n} done ({time.time()-t0:.0f}s)", flush=True)
main6 = pd.DataFrame(rows); main6.to_csv(f"{OUT}/raw/main6_results.csv", index=False)

# ---------------- B2-B4: robustness studies
SW = {}
SW["int8"] = sweep4("int8", [0, 8], ["Static", "Online-SGD", "RandomSkip", "CEGA-Tiny", "CEGA-Lite"], lambda v: ({"quant_bits": v}, None))
SW["delay"] = sweep4("delay", [0, 1, 3, 5], ["Online-SGD", "RandomSkip", "CEGA-Tiny", "CEGA-Lite"], lambda v: ({"label_delay": v}, None))
SW["solar"] = sweep4("solar", [(p, b) for p in ("const", "solar") for b in (0.1, 0.25, 0.5)],
                     ["Online-SGD", "Periodic-Full", "CEGA-Tiny", "CEGA-Lite", "CEGA-noSoC"],
                     lambda v: ({"harvest_profile": v[0], "beta": v[1]}, None))

print("experiments finished", f"{time.time()-t0:.0f}s")


main GasDrift done (35s)
main UCI-HAR done (93s)
main Elec2 done (138s)
main CoverType done (209s)
main SEA done (263s)
main Weather done (283s)
sweep int8    : 180 runs in 64s
sweep delay   : 288 runs in 97s
sweep solar   : 540 runs in 188s
experiments finished 632s


In [11]:
# ============================== B3. Part B statistics
M_MAIN = MAIN_METHODS + ["CEGA-Lite", "SB-Matched", "Entropy-Matched"]
main6 = pd.read_csv(f"{OUT}/raw/main6_results.csv")
SW = {k: pd.read_csv(f"{OUT}/raw/sweep_{k}.csv") for k in ("int8", "delay", "solar")}
# ---------------- statistics
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for k, i in enumerate(o):
        run = max(run, min(1.0, (len(p) - k) * p[i])); adj[i] = run
    return adj

def compare(df, ref, others, metric):
    piv = df.pivot_table(index=["dataset", "seed"], columns="method", values=metric)
    res = []
    for c in others:
        d = (piv[ref] - piv[c]).dropna()
        if np.allclose(d, 0): st, p = np.nan, 1.0
        else: st, p = stats.wilcoxon(d, zero_method="zsplit")
        ci = stats.t.interval(0.95, len(d) - 1, loc=d.mean(), scale=stats.sem(d)) if d.std() > 0 else (d.mean(), d.mean())
        res.append(dict(metric=metric, comparison=f"{ref} vs {c}", n=len(d), mean_diff=d.mean(), ci_low=ci[0],
                        ci_high=ci[1], d_z=(d.mean() / d.std() if d.std() > 0 else np.nan), W=st, p=p))
    out = pd.DataFrame(res); out["p_holm"] = holm(out.p.values); return out

st = []
for metric in ["acc", "macro_f1", "E_adapt_mJ"]:
    piv = main6[main6.method.isin(MAIN_METHODS)].pivot_table(index=["dataset", "seed"], columns="method", values=metric)[MAIN_METHODS]
    fr = stats.friedmanchisquare(*[piv[c].values for c in piv.columns])
    ranks = piv.rank(axis=1, ascending=(metric == "E_adapt_mJ")).mean().sort_values()
    st.append(pd.DataFrame([dict(metric=metric, comparison="Friedman (8 main methods)", n=len(piv), W=fr.statistic, p=fr.pvalue,
                                 note="; ".join(f"{k}={v:.2f}" for k, v in ranks.items()))]))
    st.append(compare(main6, "CEGA-Tiny", [m for m in M_MAIN if m != "CEGA-Tiny"], metric))
    st.append(compare(main6, "CEGA-Lite", ["Online-SGD", "RandomSkip", "SB-Matched", "Entropy-Matched"], metric))
stat6 = pd.concat(st, ignore_index=True); stat6.to_csv(f"{OUT}/tables/statistics6.csv", index=False)

import statsmodels.formula.api as smf
mm = []
for ref, oth in [("CEGA-Tiny", "Online-SGD"), ("CEGA-Tiny", "Periodic-Full"), ("CEGA-Tiny", "RandomSkip"),
                 ("CEGA-Tiny", "SB-Matched"), ("CEGA-Tiny", "Entropy-Matched"), ("CEGA-Lite", "Online-SGD"), ("CEGA-Lite", "RandomSkip")]:
    for metric, scale in [("acc", 100.0), ("E_adapt_mJ", 1.0)]:
        piv = main6.pivot_table(index=["dataset", "seed"], columns="method", values=metric)
        d = (scale * (piv[ref] - piv[oth])).rename("diff").reset_index()
        fit = smf.mixedlm("diff ~ 1", d, groups=d["dataset"]).fit(reml=True)
        ci = fit.conf_int().loc["Intercept"]
        fav = int((d.groupby("dataset")["diff"].mean() > 0).sum()) if metric == "acc" else int((d.groupby("dataset")["diff"].mean() < 0).sum())
        mm.append(dict(comparison=f"{ref} vs {oth}", metric=metric, estimate=fit.params["Intercept"], ci_low=ci[0], ci_high=ci[1],
                       p=fit.pvalues["Intercept"], datasets_favouring_first=f"{fav}/{d.dataset.nunique()}"))
mixed6 = pd.DataFrame(mm); mixed6.to_csv(f"{OUT}/tables/mixed6.csv", index=False)
for k, df in SW.items():
    st2 = []
    for v, g in df.groupby("value"):
        if "RandomSkip" in g.method.unique():
            st2.append(compare(g, "CEGA-Tiny", ["RandomSkip", "Online-SGD"], "acc").assign(value=v))
    if st2: pd.concat(st2).to_csv(f"{OUT}/tables/statistics_{k}.csv", index=False)
print("stats done")


/usr/local/lib/python3.13/dist-packages/statsmodels/regression/mixed_linear_model.py:2384: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  rslt = super().fit(
/tmp/ipykernel_1006/2821494707.py:42: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  fit = smf.mixedlm("diff ~ 1", d, groups=d["dataset"]).fit(reml=True)
/usr/local/lib/python3.13/dist-packages/statsmodels/regression/mixed_linear_model.py:2384: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  rslt = super().fit(
/tmp/ipykernel_1006/2821494707.py:42: ConvergenceWarning: Retrying MixedLM optimization with cg
  fit = smf.mixedlm("diff ~ 1", d, groups=d["dataset"]).fit(reml=True)
/usr/local/lib/python3.13/dist-packages/statsmodels/regression/mixed_linear_model.py:2384: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  rslt = super().fit(
/tmp/ipykernel_1006/2821494707.py:42: ConvergenceWarni

stats done


In [12]:
# ============================== B4. Part B figures
plt.rcParams.update({"font.size": 8, "axes.spines.top": False, "axes.spines.right": False})
COL = {"Online-SGD": "#1f77b4", "RandomSkip": "#bcbd22", "CEGA-Tiny": "#d62728", "CEGA-Lite": "#ff9896",
       "SB-Matched": "#9467bd", "Entropy-Matched": "#8c564b", "Periodic-Full": "#ff7f0e", "CEGA-noSoC": "#2ca02c", "Static": "#7f7f7f"}
def save(fig, name):
    fig.tight_layout(); fig.savefig(f"{OUT}/figures/{name}.pdf"); fig.savefig(f"{OUT}/figures/{name}.png", dpi=300); plt.close(fig)

m = pd.read_csv(f"{OUT}/raw/main6_results.csv")
meths = ["Online-SGD", "RandomSkip", "SB-Matched", "Entropy-Matched", "CEGA-Tiny", "CEGA-Lite"]
ds = ["GasDrift", "UCI-HAR", "Elec2", "CoverType", "SEA", "Weather"]
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.6)); x = np.arange(len(ds)); w = 0.13
for k, me in enumerate(meths):
    a = [m[(m.dataset == d) & (m.method == me)].acc.mean() for d in ds]
    e = [m[(m.dataset == d) & (m.method == me)].E_adapt_mJ.mean() for d in ds]
    axs[0].bar(x + k * w, a, w, color=COL[me], label=me); axs[1].bar(x + k * w, e, w, color=COL[me])
for ax, t in zip(axs, ["Prequential accuracy", "Adaptation energy (mJ)"]):
    ax.set_xticks(x + 2.5 * w); ax.set_xticklabels(ds, fontsize=7, rotation=20); ax.set_title(t)
axs[0].set_ylim(0.65, 0.92); axs[0].legend(fontsize=6, ncol=2, frameon=False)
save(fig, "figB1_selection_baselines")

def line(df, xname, fname, xlabel, methods, xlabels=None):
    fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.5))
    vals = list(dict.fromkeys(df.value))
    for me in methods:
        g = df[df.method == me].groupby("value")[["acc", "E_adapt_mJ"]].mean().reindex(vals)
        axs[0].plot(range(len(vals)), g.acc, "o-", color=COL[me], label=me, ms=3)
        axs[1].plot(range(len(vals)), g.E_adapt_mJ, "o-", color=COL[me], ms=3)
    for ax, t in zip(axs, ["Accuracy (mean of 6 streams)", "Adaptation energy (mJ)"]):
        ax.set_xticks(range(len(vals))); ax.set_xticklabels(xlabels or vals); ax.set_xlabel(xlabel); ax.set_title(t)
    axs[0].legend(fontsize=6, frameon=False); save(fig, fname)

line(pd.read_csv(f"{OUT}/raw/sweep_delay.csv"), "value", "figB2_label_delay", "Label delay (windows)",
     ["Online-SGD", "RandomSkip", "CEGA-Tiny", "CEGA-Lite"])
line(pd.read_csv(f"{OUT}/raw/sweep_int8.csv"), "value", "figB3_int8", "Inference precision",
     ["Online-SGD", "RandomSkip", "CEGA-Tiny", "CEGA-Lite"], ["float32", "int8"])
s = pd.read_csv(f"{OUT}/raw/sweep_solar.csv")
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.5)); vals = list(dict.fromkeys(s.value))
for me in ["Online-SGD", "Periodic-Full", "CEGA-noSoC", "CEGA-Tiny", "CEGA-Lite"]:
    g = s[s.method == me].groupby("value")[["violations", "E_adapt_mJ"]].mean().reindex(vals)
    axs[0].plot(range(len(vals)), g.violations, "o-", color=COL[me], label=me, ms=3)
    axs[1].plot(range(len(vals)), g.E_adapt_mJ, "o-", color=COL[me], ms=3)
lab = [v.replace("'", "").replace("(", "").replace(")", "").replace(", ", "\nβ=") for v in vals]
for ax, t in zip(axs, ["Budget violations per stream", "Adaptation energy (mJ)"]):
    ax.set_xticks(range(len(vals))); ax.set_xticklabels(lab, fontsize=6); ax.set_title(t)
axs[0].legend(fontsize=6, frameon=False); save(fig, "figB4_solar_harvest")
print("figures written")


figures written


In [13]:
# ============================== B5. Package and download
json.dump(dict(version="v4-partB", numpy=np.__version__, python=sys.version.split()[0]), open(f"{OUT}/config_partB.json", "w"), indent=2)
shutil.make_archive(OUT, "zip", OUT); print("Packaged:", OUT + ".zip")
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(OUT + ".zip")


Packaged: cega_tiny_v5_10seed_results.zip
download skipped


In [14]:
import pandas as pd; m=pd.read_csv(f"{OUT}/raw/main6_results.csv"); g=m.groupby(['dataset','method']).agg(acc=('acc','mean'),acc_sd=('acc','std'),f1=('macro_f1','mean'),f1_sd=('macro_f1','std'),E=('E_adapt_mJ','mean'),E_sd=('E_adapt_mJ','std'),upd=('frac_updated','mean'),viol=('violations','mean'),rec=('recovery_win','mean')).round(4); print('###MAIN'); print(g.to_csv()); _=[print('###'+f+'\n'+pd.read_csv(f'{OUT}/tables/{f}.csv').round(5).to_csv(index=False)) for f in ['statistics6','mixed6','statistics_delay','statistics_int8']]; _=[print('###sweep_'+k+'\n'+pd.read_csv(f'{OUT}/raw/sweep_{k}.csv').groupby(['dataset','method','value'])[['acc','E_adapt_mJ','violations','frac_updated']].mean().round(4).to_csv()) for k in ['int8','delay','solar']]

###MAIN
dataset,method,acc,acc_sd,f1,f1_sd,E,E_sd,upd,viol,rec
CoverType,CEGA-Lite,0.8323,0.0032,0.6594,0.0121,24.5344,0.7345,0.5073,0.0,
CoverType,CEGA-Tiny,0.8311,0.0038,0.6547,0.0137,30.2574,2.6504,0.5697,0.0,
CoverType,DriftTrig-Full,0.7969,0.0115,0.6014,0.0248,67.5006,11.3516,0.0,0.1,
CoverType,Entropy-Matched,0.834,0.003,0.6592,0.0119,27.9345,0.8362,0.5701,0.0,
CoverType,GovRetrain,0.7609,0.0567,0.5417,0.0583,67.1473,28.0016,0.0,0.0,
CoverType,Online-SGD,0.8283,0.0036,0.6562,0.0125,44.1731,0.3753,1.0,0.0,
CoverType,Periodic-Full,0.826,0.0041,0.6033,0.0122,199.9337,0.0667,0.0,43.4,
CoverType,RandomSkip,0.8231,0.0034,0.6444,0.011,27.8493,0.8863,0.5681,0.0,
CoverType,SB-Matched,0.8283,0.0037,0.6529,0.0121,27.9345,0.8362,0.5701,0.0,
CoverType,Static,0.6992,0.0315,0.5074,0.0201,0.0,0.0,0.0,0.0,
CoverType,TinyOL-Head,0.784,0.0065,0.5912,0.0139,16.6254,0.1626,1.0,0.0,
Elec2,CEGA-Lite,0.7323,0.0113,0.7246,0.0123,10.827,0.7085,0.5837,0.0,
Elec2,CEGA-Tiny,0.7338,0.0115,0.7259,0.0128,13.926

In [15]:
import pandas as pd; R=lambda d:d.round(4).to_csv(index=False).replace('=',':').replace(';','|'); m=pd.read_csv(f"{OUT}/raw/main6_results.csv"); g=m.groupby(['dataset','method']).agg(acc=('acc','mean'),asd=('acc','std'),f1=('macro_f1','mean'),fsd=('macro_f1','std'),E=('E_adapt_mJ','mean'),Esd=('E_adapt_mJ','std'),upd=('frac_updated','mean'),viol=('violations','mean'),rec=('recovery_win','mean')).reset_index(); g=g[g.dataset.isin(['SEA','Weather'])|g.method.isin(['SB-Matched','Entropy-Matched'])]; s=pd.read_csv(f"{OUT}/tables/statistics6.csv")[['metric','comparison','mean_diff','ci_low','ci_high','d_z','p_holm','note']]; x=pd.read_csv(f"{OUT}/tables/mixed6.csv"); sw=pd.concat([pd.read_csv(f"{OUT}/raw/sweep_{k}.csv").assign(sw=k) for k in ['int8','delay','solar']]); a=sw.groupby(['sw','method','value'])[['acc','E_adapt_mJ','violations','frac_updated']].mean().reset_index(); t=pd.concat([pd.read_csv(f"{OUT}/tables/statistics_{k}.csv").assign(sw=k) for k in ['delay','int8']])[['sw','value','comparison','mean_diff','p_holm']]; print('#G\n'+R(g)+'#S\n'+R(s)+'#X\n'+R(x)+'#A\n'+R(a)+'#T\n'+R(t))

#G
dataset,method,acc,asd,f1,fsd,E,Esd,upd,viol,rec
CoverType,Entropy-Matched,0.834,0.003,0.6592,0.0119,27.9345,0.8362,0.5701,0.0,
CoverType,SB-Matched,0.8283,0.0037,0.6529,0.0121,27.9345,0.8362,0.5701,0.0,
Elec2,Entropy-Matched,0.7321,0.0111,0.7238,0.0122,13.1579,0.5563,0.7347,0.0,
Elec2,SB-Matched,0.7333,0.0112,0.7255,0.0121,13.1579,0.5563,0.7347,0.0,
GasDrift,Entropy-Matched,0.8382,0.0098,0.83,0.0106,9.1729,0.3786,0.466,0.0,10.1429
GasDrift,SB-Matched,0.8418,0.0103,0.8339,0.0109,9.1729,0.3786,0.466,0.0,11.0857
SEA,CEGA-Lite,0.8531,0.004,0.8412,0.0044,20.3867,0.3569,0.9204,0.0,7.7667
SEA,CEGA-Tiny,0.857,0.0044,0.8457,0.0048,22.0578,0.6552,0.9193,0.0,7.9667
SEA,DriftTrig-Full,0.8554,0.0073,0.8438,0.0082,10.5844,3.2207,0.0,0.0,7.6667
SEA,Entropy-Matched,0.8551,0.0042,0.8435,0.0046,20.3661,0.3648,0.9191,0.0,7.7667
SEA,GovRetrain,0.8485,0.0065,0.8359,0.0085,7.3256,1.7628,0.0,0.0,5.4
SEA,Online-SGD,0.8533,0.0037,0.8414,0.0041,21.9186,0.1776,1.0,0.0,7.5667
SEA,Periodic-Full,0.8716,0.0032,0

In [16]:
exec(__import__('base64').b64decode('aW1wb3J0IHBhbmRhcyBhcyBwZCwgbnVtcHkgYXMgbnAKbSA9IHBkLnJlYWRfY3N2KGYie09VVH0vcmF3L21haW42X3Jlc3VsdHMuY3N2IikKZyA9IG0uZ3JvdXBieShbImRhdGFzZXQiLCJtZXRob2QiXSkuYWdnKG49KCJhY2MiLCJzaXplIiksYWNjPSgiYWNjIiwibWVhbiIpLGFzZD0oImFjYyIsInN0ZCIpLGYxPSgibWFjcm9fZjEiLCJtZWFuIiksRT0oIkVfYWRhcHRfbUoiLCJtZWFuIiksRXNkPSgiRV9hZGFwdF9tSiIsInN0ZCIpLHVwZD0oImZyYWNfdXBkYXRlZCIsIm1lYW4iKSx2aW9sPSgidmlvbGF0aW9ucyIsInN1bSIpKS5yZXNldF9pbmRleCgpCnByaW50KCIjIyNNQUlOMTAiKTsgcHJpbnQoZy5yb3VuZCg1KS50b19jc3YoaW5kZXg9RmFsc2UpKQpmb3IgZHMgaW4gWyJXZWF0aGVyIiwiU0VBIl06CiAgICB3ID0gbVttLmRhdGFzZXQ9PWRzXS5waXZvdF90YWJsZShpbmRleD0ic2VlZCIsIGNvbHVtbnM9Im1ldGhvZCIsIHZhbHVlcz0iYWNjIikKICAgIGQgPSAod1siQ0VHQS1UaW55Il0gLSB3WyJSYW5kb21Ta2lwIl0pCiAgICBwcmludCgiIyMjIitkcysiIENFR0EtUmFuZG9tU2tpcCIsICJtZWFuIiwgcm91bmQoZC5tZWFuKCksNiksICJXL1QvTCIsIGludCgoZD4wKS5zdW0oKSksIGludCgoZD09MCkuc3VtKCkpLCBpbnQoKGQ8MCkuc3VtKCkpKQpyID0gbS5ncm91cGJ5KFsiZGF0YXNldCIsInNlZWQiLCJtZXRob2QiXSkuYWNjLm1lYW4oKS5yZXNldF9pbmRleCgpOyByWyJyYW5rIl0gPSByLmdyb3VwYnkoWyJkYXRhc2V0Iiwic2VlZCJdKS5hY2MucmFuayhhc2NlbmRpbmc9RmFsc2UpCnByaW50KCIjIyNSQU5LIik7IHByaW50KHIuZ3JvdXBieSgibWV0aG9kIilbInJhbmsiXS5tZWFuKCkuc29ydF92YWx1ZXMoKS5yb3VuZCgzKS50b19jc3YoKSkKZm9yIGYgaW4gWyJzdGF0aXN0aWNzNiIsIm1peGVkNiJdOgogICAgcHJpbnQoIiMjIyIrZik7IHByaW50KHBkLnJlYWRfY3N2KGYie09VVH0vdGFibGVzL3tmfS5jc3YiKS5yb3VuZCg1KS50b19jc3YoaW5kZXg9RmFsc2UpKQpwcmludCgiIyMjRU5EIikK').decode())

###MAIN10
dataset,method,n,acc,asd,f1,E,Esd,upd,viol
CoverType,CEGA-Lite,10,0.83226,0.00318,0.65944,24.53436,0.73452,0.50734,0
CoverType,CEGA-Tiny,10,0.83111,0.00381,0.65467,30.2574,2.65045,0.56974,0
CoverType,DriftTrig-Full,10,0.79691,0.01148,0.60139,67.5006,11.35155,0.0,1
CoverType,Entropy-Matched,10,0.83401,0.00296,0.65925,27.93453,0.83622,0.57006,0
CoverType,GovRetrain,10,0.76093,0.0567,0.54171,67.14732,28.00158,0.0,0
CoverType,Online-SGD,10,0.82829,0.00363,0.6562,44.17307,0.3753,1.0,0
CoverType,Periodic-Full,10,0.82604,0.00413,0.60325,199.93371,0.06672,0.0,434
CoverType,RandomSkip,10,0.82307,0.00335,0.64439,27.84932,0.8863,0.56809,0
CoverType,SB-Matched,10,0.82833,0.00368,0.65294,27.93453,0.83622,0.57006,0
CoverType,Static,10,0.69924,0.03148,0.50743,0.0,0.0,0.0,0
CoverType,TinyOL-Head,10,0.78396,0.00654,0.59116,16.62535,0.16258,1.0,0
Elec2,CEGA-Lite,10,0.73228,0.01135,0.72463,10.82704,0.70853,0.58372,0
Elec2,CEGA-Tiny,10,0.73382,0.0115,0.72588,13.92663,0.52438,0.73411,0
Elec2,Drif